# 📓 Semana 7 · Dia 2 — Segurança: GRANT, RLS e Column Masking

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (UC), DEP (governança) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | RLS + masking aplicados no Ouro |

---


## 📖 Teoria — O modelo de permissões do UC

Permissões são concedidas em qualquer nível: catálogo, schema, tabela/view, coluna, volume, função.

```sql
GRANT SELECT ON TABLE workspace.ouro.vendas_por_dia TO analistas;
GRANT USAGE ON SCHEMA workspace.ouro TO analistas;
REVOKE ...
```

Grupos (analistas, engenheiros, admins) centralizam o acesso — nunca conceda por usuário direto.


## 📖 Teoria — RLS e Column Masking

**Row-Level Security (RLS)**: filtra **linhas** por usuário — um vendedor vê só as vendas da sua região.
**Column Masking**: mascara **colunas** — CPF/email aparecem mascarados para quem não tem permissão.

Implementação: **dynamic view** (view com `current_user()` no WHERE) ou **functions de masking** declaradas no UC.


### 💻 Na prática — Dynamic view com RLS

Crie uma view que filtra linhas por usuário — na Free Edition, o padrão para proteger o Ouro.


In [ ]:
# Função auxiliar: mapear usuário -> país permitido
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.ouro.pais_do_usuario()
RETURNS STRING
RETURN CASE WHEN current_user() = 'ana@empresa.com' THEN 'United Kingdom'
            WHEN current_user() = 'joao@empresa.com' THEN 'BRAZIL'
            ELSE '*' END
""")
print("Função de mapeamento usuário->país criada.")

In [ ]:
# Dynamic view com RLS (linhas) + masking (coluna)
spark.sql("""
CREATE OR REPLACE VIEW workspace.ouro.vendas_rls_vw AS
SELECT InvoiceNo,
       CASE WHEN current_user() IN ('admin@empresa.com') THEN CustomerID
            ELSE concat(substring(CustomerID, 1, 2), '***') END AS CustomerID,
       Country, Quantity, UnitPrice, Quantity * UnitPrice AS receita
FROM workspace.bronze.vendas_bronze
WHERE '*' = (SELECT pais_do_usuario())
   OR UPPER(Country) = (SELECT pais_do_usuario())
""")
print("Dynamic view com RLS + masking criada.")

In [ ]:
# Testar como o usuário atual
display(spark.sql("SELECT * FROM workspace.ouro.vendas_rls_vw LIMIT 10"))
print("Se você não é admin, o CustomerID vem mascarado e o filtro de país aplica.")

### 💻 Na prática — Permissões na prática

Conceda e revogue acesso — e teste o efeito.


In [ ]:
%sql
GRANT USAGE ON SCHEMA workspace.ouro TO `account users`;
GRANT SELECT ON TABLE workspace.ouro.vendas_rls_vw TO `account users`;
SHOW GRANTS ON TABLE workspace.ouro.vendas_rls_vw;

> 🎯 **Dica de prova**: DEA/DEP cobram: GRANT/REVOKE por nível, dynamic views com current_user para RLS/masking, e a diferença entre line-level e column-level security. Memize os exemplos.


## 🎯 Exercícios de fixação

**1.** Crie uma dynamic view que mascara o e-mail do cliente (ex.: ana@ → a***@).

**2.** Como aplicar RLS por grupo e não por usuário?

**3.** O que acontece se um usuário sem SELECT consultar a view?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Masking de e-mail

```sql
CASE WHEN current_user() = 'admin' THEN email
     ELSE concat(substring(email,1,1),'***@',split(email,'@')[1]) END
```

**2.** Por grupo

Use `is_account_group_member('grupo')` no CASE/WHERE — a dynamic view avalia o grupo em vez do usuário individual.

**3.** Sem SELECT

Falha de permissão (erro). A view exige privilégios próprios + os dos objetos de baixo — o UC exige GRANT explícito.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*